<a href="https://colab.research.google.com/github/oliverclaypole/Interior_Allocations/blob/main/Interior_Allocations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import pandas as pd
from datetime import datetime
from collections import defaultdict

In [76]:
# 1. Mock Database
# Mock SQL tables.
staff_data = [
    {"id": 1, "name": "Alex", "skill": "Lead Designer", "day_rate": 300, "contracted_hours": 40},
    {"id": 2, "name": "Jordan", "skill": "Painter", "day_rate": 180, "contracted_hours": 35},
    {"id": 3, "name": "Sam", "skill": "Carpenter", "day_rate": 220, "contracted_hours": 40},
    {"id": 4, "name": "Peter", "skill": "Labourer", "day_rate": 150, "contracted_hours": 40}
]

shift_ledger = [
    {"worker_id": 1, "project": "Mayfair Penthouse", "hours": 8, "date": "2026-04-25"},
    {"worker_id": 1, "project": "Mayfair Penthouse", "hours": 8, "date": "2026-04-26"},
    {"worker_id": 2, "project": "Chelsea Studio", "hours": 10, "date": "2026-04-26"}
]

In [1]:
# Calculations Staff Hours
def get_staff_summary(worker_id):

    person = next(item for item in staff_data if item["id"] == worker_id)
    worked = sum(shift['hours'] for shift in shift_ledger if shift['worker_id'] == worker_id)
    difference = person['contracted_hours'] - worked
    status = "OWES HOURS" if difference > 0 else "TARGET MET"

    # Calculate Pay Owed (Rate / 8 hours per day)
    pay_owed = worked * (person['day_rate'] / 8)

    return {
        "Name": person['name'],
        "Worked": worked,
        "Contract": person['contracted_hours'],
        "Difference": abs(difference),
        "Status": status,
        "Current_Pay": f"£{pay_owed:,.2f}"
    }

def calculate_shift_duration(start_time, end_time):
  fmt = "%H:%M"
  start = datetime.strptime(start_time, fmt)
  end = datetime.strptime(end_time, fmt)
  duration = end - start
  return duration.total_seconds() / 3600

def check_rota_conflict(worker_id, date):
    # Search the ledger for any shift for this worker on this date
    conflicts = [s for s in shift_ledger if s['worker_id'] == worker_id and s['date'] == date]

    if conflicts:
        print(f"⚠️ ROTA CONFLICT: This worker is already assigned to '{conflicts[0]['project']}' on {date}.")
        return True
    return False

# Temporary Ledger
if 'ledger' not in globals():
    ledger = []
                                # in the real app times come from clock widgets
def ui_add_shift_form(worker_name, start_val, end_val):
  fmt = "%H:%M"
  start = datetime.strptime(start_val, fmt)
  end = datetime.strptime(end_val, fmt)
  duration = (end - start).total_seconds()/3600

  if duration <=0:
    return "Finish time must be after start time"

  # creating record
  new_entry = {
        "worker": worker_name,
        "start": start_val,
        "end": end_val,
        "duration": round(duration, 2),
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  }
  ledger.append(new_entry)
  return f"Success! Logged {round(duration, 2)} hours for {worker_name}."

In [78]:
# 1. We need a list of "Standard Rates" for roles
role_rates = {
    "Labourer": 150,   # Day rate
    "Painter": 180,
    "Carpenter": 220,
    "Lead Designer": 300
}

estimated_project_costs = {} # Global dictionary to store calculated estimates

def create_project_estimate(project_name, staff_requirements):
    """
    staff_requirements is a list of tuples: (Role, Days)
    Example: [("Labourer", 4), ("Painter", 0.5)]
    """
    print(f"\n--- ESTIMATE FOR: {project_name.upper()} ---")
    total_estimate = 0

    for role, days in staff_requirements:
        # Get the rate for that role
        rate = role_rates.get(role, 0)
        cost = rate * days
        total_estimate += cost

        # Friendly breakdown
        unit = "days" if days >= 1 else "day (half-day/hours)"
        print(f"• {role}: {days} {unit} @ £{rate}/day = £{cost:,.2f}")

    print("-" * 30)
    print(f"TOTAL ESTIMATED STAFF COST: £{total_estimate:,.2f}")
    print("-" * 30)
    estimated_project_costs[project_name.lower()] = total_estimate # Store the estimate
    return total_estimate

# --- DAD'S INPUT EXAMPLE ---
# "5 labourers for 4 days" and "2 decorators (Painters) for 2 days"
requirements = [
    ("Labourer", 5 * 4),    # 5 people for 4 days = 20 total days of labour
    ("Painter", 2 * 2),     # 2 people for 2 days = 4 total days of painting
    ("Carpenter", 0.5)      # Plus a half-day for a quick fix
]

estimate_total = create_project_estimate("Modern Apartment Refurb", requirements)


--- ESTIMATE FOR: MODERN APARTMENT REFURB ---
• Labourer: 20 days @ £150/day = £3,000.00
• Painter: 4 days @ £180/day = £720.00
• Carpenter: 0.5 day (half-day/hours) @ £220/day = £110.00
------------------------------
TOTAL ESTIMATED STAFF COST: £3,830.00
------------------------------


In [79]:
# 1. Project Estimates
project_blueprints = {
    "Chelsea Studio": [
        {"role": "Painter", "quantity": 4, "days": 4},
        {"role": "Labourer", "quantity": 1, "days": 2}
    ]
}

def view_project_plan(project_name):
    if project_name not in project_blueprints:
        print("Project plan not found.")
        return

    print(f"\n--- ORIGINAL PLAN FOR: {project_name.upper()} ---")
    plan = project_blueprints[project_name]

    total_estimated_cost = 0
    for item in plan:
        # Calculate cost based on role rates we defined earlier
        rate = role_rates.get(item['role'], 0)
        cost = item['quantity'] * item['days'] * rate
        total_estimated_cost += cost

        # This is the 'Note' that is generated automatically
        print(f"Plan: {item['quantity']}x {item['role']}(s) for {item['days']} days @ £{rate}/day = £{cost:,.2f}")

    print(f"TOTAL PLAN COST: £{total_estimated_cost:,.2f}")

# Let's see the plan he made
view_project_plan("Chelsea Studio")


--- ORIGINAL PLAN FOR: CHELSEA STUDIO ---
Plan: 4x Painter(s) for 4 days @ £180/day = £2,880.00
Plan: 1x Labourer(s) for 2 days @ £150/day = £300.00
TOTAL PLAN COST: £3,180.00


In [80]:
# --- INTERACTIVE PROJECT STAFF REQUIREMENT ESTIMATION ---
print("\n--- INTERACTIVE PROJECT STAFF ESTIMATION ---")

while True:
    estimate_project_name = input("Enter project name for estimation (or 'done' to finish): ").strip()
    if estimate_project_name.lower() == 'done':
        break

    print(f"\n--- Collecting staff requirements for '{estimate_project_name}' ---")
    current_requirements = []
    while True:
        role_input = input("Enter staff role (e.g., 'Labourer', 'Painter') or 'done' to finish roles: ").strip()
        if role_input.lower() == 'done':
            break

        matched_role = None
        for key in role_rates.keys():
            if key.lower() == role_input.lower():
                matched_role = key # Use the exact key from role_rates
                break

        if not matched_role:
            print(f"❌ Role '{role_input}' not recognized. Please choose from: {', '.join(role_rates.keys())}")
            continue

        try:
            quantity = int(input(f"Enter quantity for {matched_role}: "))
            days = float(input(f"Enter number of days for {matched_role}: "))

            if quantity <= 0 or days <= 0:
                print("❌ Quantity and Days must be positive numbers.")
                continue

            total_days_for_role = quantity * days
            current_requirements.append((matched_role, total_days_for_role))
        except ValueError:
            print("❌ Invalid number for quantity or days. Please enter numerical values.")

    if current_requirements:
        create_project_estimate(estimate_project_name, current_requirements)
    else:
        print("No staff requirements entered for this project.")
    print("-" * 50)

print("Interactive staff estimation session ended.")


--- INTERACTIVE PROJECT STAFF ESTIMATION ---


KeyboardInterrupt: Interrupted by user

In [ ]:
# 3. Test Run For Alex
print("--- MANAGER'S DASHBOARD VIEW ---")
alex_report = get_staff_summary(1)
print(pd.Series(alex_report))

In [ ]:
# Input Logic
def log_new_hours():
    print("\n--- NEW SHIFT ENTRY FORM ---")

    # 1. IDENTIFY THE WORKER
    new_worker_input = input("Enter worker ID or Name: ")
    new_worker_id = None
    worker_name = ""

    # Check ID or Name logic
    try:
        new_worker_id = int(new_worker_input)
        # Find the name that goes with this ID
        match = next((s for s in staff_data if s['id'] == new_worker_id), None)
        if match:
            worker_name = match['name']
    except ValueError:
        # It's a name, find the ID
        match = next((s for s in staff_data if s['name'].lower() == new_worker_input.lower()), None)
        if match:
            new_worker_id = match['id']
            worker_name = match['name']

    if not new_worker_id:
        print("❌ Worker not found.")
        return

    # 2. COLLECT PROJECT AND TIME
    new_project = input("Enter project name: ").lower()
    start_t = input("Start Time (e.g. 08:00): ")
    end_t = input("End Time (e.g. 17:30): ")

    # Modify date input here
    while True:
        date_input = input("Enter date (DD-MM, e.g., 28-04, for year 2026): ")
        if len(date_input) == 5 and date_input[2] == '-' and date_input[:2].isdigit() and date_input[3:].isdigit():
            day_str, month_str = date_input.split('-')
            new_date = f"2026-{month_str}-{day_str}"
            try:
                datetime.strptime(new_date, "%Y-%m-%d") # Validate the full date
                break
            except ValueError:
                print("❌ Error: Invalid date format or value for DD-MM.")
        else:
            print("❌ Error: Please enter the date in DD-MM format (e.g., 28-04).")

    # 3. ROTA CONFLICT CHECK
    if check_rota_conflict(new_worker_id, new_date):
        print("❌ Entry blocked to prevent double-booking.")
        return

    # 4. CALCULATE HOURS (The "Don't make them do math" part)
    try:
        calculated_hours = calculate_shift_duration(start_t, end_t)
        if calculated_hours <= 0:
            print("❌ Error: Finish time must be after start time.")
            return
    except ValueError:
        print("❌ Error: Use the HH:MM format (e.g. 09:00).")
        return

    # 5. SAVE TO THE MAIN LEDGER (Using ID for the database, but we know the name)
    new_shift = {
        "worker_id": new_worker_id,
        "project": new_project,
        "hours": round(calculated_hours, 2),
        "date": new_date,
        "start": start_t,
        "end": end_t
    }

    shift_ledger.append(new_shift)

    # 6. FEEDBACK (Easy on the eyes)
    print(f"\n✅ SUCCESS!")
    print(f"Logged {round(calculated_hours, 2)} hours for {worker_name} on the {new_project} project.")

# Now run it
log_new_hours()

# Temporary Ledger
if 'ledger' not in globals():
    ledger = []
                                # in the real app times come from clock widgets
def ui_add_shift_form(worker_name, start_val, end_val):
  fmt = "%H:%M"
  start = datetime.strptime(start_val, fmt)
  end = datetime.strptime(end_val, fmt)
  duration = (end - start).total_seconds()/3600

  if duration <=0:
    return "Finish time must be after start time"

  # creating record
  new_entry = {
        "worker": worker_name,
        "start": start_val,
        "end": end_val,
        "duration": round(duration, 2),
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  }
  ledger.append(new_entry)
  return f"Success! Logged {round(duration, 2)} hours for {worker_name}."


--- NEW SHIFT ENTRY FORM ---


In [ ]:
# Creating Managers Dashboard

def view_manager_dashboard():
    print("\n" + "-"*50)
    print("        INTERIOR REVOLUTIONS'S WEEKLY OVERVIEW")
    print("-"*50)

    # Dictionary to aggregate hours and costs by worker and project
    aggregated_data = defaultdict(lambda: {"Hours": 0, "Cost_raw": 0})

    for shift in shift_ledger:
        # Find the worker's name and rate from staff_data using their ID
        worker = next(s for s in staff_data if s['id'] == shift['worker_id'])

        # Calculate cost for this specific shift
        shift_cost = shift['hours'] * (worker['day_rate'] / 8)

        # Use (worker_name, project_name) as the key for aggregation
        key = (worker['name'], shift['project'])

        # Aggregate hours and cost
        aggregated_data[key]['Worker'] = worker['name']
        aggregated_data[key]['Project'] = shift['project']
        aggregated_data[key]['Hours'] += shift['hours']
        aggregated_data[key]['Cost_raw'] += shift_cost

    # Convert aggregated_data into a list of dictionaries for DataFrame creation
    report_list = []
    for item in aggregated_data.values():
        report_list.append({
            "Worker": item['Worker'],
            "Project": item['Project'],
            "Hours": item['Hours'],
            "Cost": f"£{item['Cost_raw']:,.2f}" # Format cost here
        })

    # 2. Use Pandas to make it look like a clean table
    df = pd.DataFrame(report_list)

    # Sort by Worker then Project for consistency
    df = df.sort_values(by=["Worker", "Project"])

    print(df.to_string(index=False)) # index=False hides the row numbers

    # 3. Add a "Grand Total" at the bottom
    total_spend = sum(shift['hours'] * (next(s for s in staff_data if s['id'] == shift['worker_id'])['day_rate'] / 8) for shift in shift_ledger)
    print("-"*50)
    print(f"TOTAL BUSINESS OUTGOINGS: £{total_spend:,.2f}")
    print("-"*50)

# Run the review
view_manager_dashboard()

In [ ]:
shift_ledger_df = pd.DataFrame(shift_ledger)
display(shift_ledger_df.head())

In [ ]:
# 1. Project Budget Display
# In the future, this will be its own SQL table.
project_budgets = {
    "Mayfair Penthouse": 5000.00,
    "Chelsea Studio": 2500.00,
    "Modern Apartment Refurb": 5820.00
}

def view_budget_tracker():
    print("\n" + "!"*50)
    print("        PROJECT BUDGET VARIANCE REPORT")
    print("!"*50)

    summary_data = []

    for project, budget in project_budgets.items():
        # Calculate total spent on this project so far
        total_spent = 0
        shifts_logged = 0
        for shift in shift_ledger:
            # Ensure project names are compared case-insensitively
            if shift['project'].lower() == project.lower():
                # Get worker rate
                worker = next(s for s in staff_data if s['id'] == shift['worker_id'])
                shift_cost = shift['hours'] * (worker['day_rate'] / 8)
                total_spent += shift_cost
                shifts_logged += 1

        remaining = budget - total_spent

        # Handle division by zero if budget is 0 to avoid errors
        if budget > 0:
            percent_used = (total_spent / budget) * 100
        else:
            percent_used = 0 # Or handle as an error/special case

        # Determine "Health" Status
        if percent_used > 100:
            status = "🔴 OVER BUDGET"
        elif percent_used > 80:
            status = "🟡 WARNING"
        else:
            status = "🟢 HEALTHY"

        # Get the estimated staff cost for comparison
        estimated_staff_cost = estimated_project_costs.get(project.lower(), 0.0)

        summary_data.append({
            "Project": project,
            "Estimated Staff Cost": f"£{estimated_staff_cost:,.2f}",
            "Budget": f"£{budget:,.2f}",
            "Spent": f"£{total_spent:,.2f}",
            "Remaining": f"£{remaining:,.2f}",
            "Usage": f"{percent_used:.1f}%",
            "Shifts Logged": shifts_logged,
            "Status": status
        })

    df = pd.DataFrame(summary_data)
    print(df.to_string(index=False))
    print("!"*50)

# --- Interactive Project Budget Input ---
print("\n---  PROJECT BUDGET INPUT ---")
while True:
    project_name_input = input("Enter project name (or 'done' to finish): ").strip()
    if project_name_input.lower() == 'done':
        break

    # Check if we have an estimate for this project
    suggested_budget = estimated_project_costs.get(project_name_input.lower())

    budget_input_prompt = f"Enter budget for '{project_name_input}': £"
    if suggested_budget is not None:
        budget_input_prompt = f"Enter budget for '{project_name_input}' (Calculated staff cost: £{suggested_budget:,.2f}, press Enter to use this, or type new): £"

    try:
        user_input = input(budget_input_prompt)
        if user_input == '' and suggested_budget is not None:
            budget_input = suggested_budget
        else:
            budget_input = float(user_input)

        # Update or add the project budget
        project_budgets[project_name_input] = budget_input
        print(f"✅ Budget for '{project_name_input}' set to £{budget_input:,.2f}")
    except ValueError:
        print("❌ Invalid budget amount. Please enter a number.")
    print("-" * 30)

print("\nUpdated Project Budgets:")
for project, budget in project_budgets.items():
    print(f" - {project}: £{budget:,.2f}")

# Run the tracker with the potentially updated budgets
view_budget_tracker()

In [ ]:
# Display the Project Budget Variance Report with all details
view_budget_tracker()

### Mock User Login and Menu

In [ ]:
# This is a mock user login
current_user = {"name": "Dad", "role": "Manager"} # Or {"name": "Alex", "role": "Worker"}

def display_menu(user):
    print(f"\nWelcome, {user['name']}!")
    print("[1] Clock In/Out")
    print("[2] View My Shifts")

    # The 'Manager Only' Gate
    if user['role'] == "Manager":
        print("[3] --- ADMIN ONLY: Create Project Estimate ---")
        print("[4] --- ADMIN ONLY: View Budget Dashboard ---")
    else:
        print("(Admin tools hidden)")

# Try it with the Dad
display_menu(current_user)

# Try it with a worker
display_menu({"name": "Alex", "role": "Worker"})